# Directory Pipeline — Surya OCR + Alignment Review on Google Colab

This notebook runs the three enrichment steps of the directory pipeline using a free Google Colab GPU:

| Step | Flag | What it does | Output |
|------|------|-------------|--------|
| 1 | `--surya-ocr` | Neural net draws bounding boxes around every text line | `*_surya.json` |
| 2 | `--align-ocr` | Maps Gemini's transcribed text onto those boxes | `*_aligned.json` |
| 3 | `--review-alignment` | Interactive UI to fix bad matches | updates `*_aligned.json` |

After completing these steps, every entry in your CSV will have a `canvas_fragment` URL with a precise `#xywh=` bounding box — click it to open the IIIF viewer at the exact spot on the original page.

---

### Before you start

You should already have run the basic pipeline steps for your volume:
- `--download` — page images are in `output/<slug>/`
- `--gemini-ocr` — `*_gemini.txt` files exist alongside the images
- `--extract-entries` — an `entries_*.csv` exists (re-run at the end to add bboxes)

> ⚠️ **Enable a GPU runtime before running anything.**  
> Go to **Runtime → Change runtime type → T4 GPU** and click Save.  
> Surya OCR requires a GPU — on CPU it will take hours instead of minutes.

---
## Setup

Run all cells in this section once per session.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [10]:
import os
from pathlib import Path

# ── Edit these values ─────────────────────────────────────────────────────────

# Path to the directory-pipeline folder on your Google Drive.
# Common locations:
#   /content/drive/MyDrive/directory-pipeline          (synced to My Drive)
#   /content/drive/Othercomputers/My_Mac/directory-pipeline  (Mac via Drive for Desktop)
PIPELINE_DIR = "/content/drive/Othercomputers/My_Mac/directory-pipeline"

# The model name used when you ran Gemini OCR.
# Check one of your *_gemini.txt filenames — the model name is embedded in it.
MODEL = "gemini-3.1-flash-lite"

# Port for the review server. 5000 is the default.
PORT = 5000

# ─────────────────────────────────────────────────────────────────────────────

os.chdir(PIPELINE_DIR)
print("✓ Working directory:", os.getcwd())

✓ Working directory: /content/drive/Othercomputers/My_Mac/directory-pipeline


In [3]:
# Install pipeline dependencies.
# surya-ocr pulls in PyTorch at the right CUDA version — takes 2–4 min the first time.
%pip install -q google-genai pillow requests python-dotenv flask geopy pyngrok
%pip install -q "surya-ocr==0.17.1" "transformers>=4.56.1,<5.0"
print("\n✓ Dependencies installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.9/189.9 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 117.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 152.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 134.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.5/226.5 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.4/99.4 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 158.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.0/469.0 kB 42.9 MB/s eta 0:00:00

✓ Dependencies installed


### Choose your volume

Enter the path to your volume **relative to the `output/` folder**.

For NYPL collections this is usually two levels deep:
```
the_green_book_9ea5d5b0/d440c0b0-1a71-0132-7f48-58d385a7bbd0
```
For Internet Archive collections it's typically one level:
```
brewers_guide_for_the_united_states_cana_bd047d10/bd047d10
```

Not sure of the path? Run: `!find output -name "*_gemini.txt" | head -10`

In [6]:
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path

VOLUME = None

slug_widget = widgets.Text(
    value="",
    placeholder="e.g. the_green_book_9ea5d5b0/d440c0b0-1a71-0132-7f48-58d385a7bbd0",
    description="Volume path:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="700px"),
)

status_out = widgets.Output()

def _on_change(change):
    global VOLUME
    slug = change["new"].strip().strip("/")
    if not slug:
        return
    VOLUME = f"output/{slug}"
    vol_path = Path(PIPELINE_DIR) / VOLUME
    with status_out:
        status_out.clear_output()
        images  = list(vol_path.glob("*.jpg")) + list(vol_path.glob("*.png"))
        gemini  = list(vol_path.glob("*_gemini.txt"))
        surya   = list(vol_path.glob("*_surya.json"))
        aligned = list(vol_path.glob("*_aligned.json"))
        print(f"Volume: {VOLUME}")
        print(f"  {len(images):4d} page images")
        print(f"  {len(gemini):4d} Gemini OCR files  (*_gemini.txt)")
        print(f"  {len(surya):4d} Surya files        (*_surya.json)  — Step 1 output")
        print(f"  {len(aligned):4d} Aligned files      (*_aligned.json) — Step 2 output")
        if not vol_path.exists():
            print("\n  ⚠ Path not found — check the volume path above")
        elif not images:
            print("\n  ⚠ No images found — check that download completed")
        elif not gemini:
            print("\n  ⚠ No Gemini OCR files — run --gemini-ocr before this step")

slug_widget.observe(_on_change, names="value")
display(slug_widget, status_out)

Text(value='', description='Volume path:', layout=Layout(width='700px'), placeholder='e.g. the_green_book_9ea5…

Output()

---
## Step 1: Surya OCR — detect bounding boxes

Gemini reads the text on each page but doesn't know *where* on the page each line sits. Surya fills that gap: it's a neural network that looks at the page image and draws a bounding box around every line of text it finds.

After this step, each page will have a `*_surya.json` file containing `(x1, y1, x2, y2)` boxes — one per detected text line. Those boxes are what get attached to entries in your final CSV.

**Batch size guidance** — higher is faster, but risks out-of-memory errors:

| GPU | Photographic / book scans | Microfilm / high-contrast |
|-----|--------------------------|---------------------------|
| T4 (Colab free tier) | 8–16 | 32–64 |
| L4 (Colab paid) | 32–72 | 72–128 |

Start with 16. If you get a CUDA out-of-memory error, lower `--batch-size` and re-run.  
**Surya skips pages that already have a `*_surya.json`**, so no work is lost if you need to retry.

**Expected time:** A few seconds per page. A 100-page volume takes roughly 5–15 minutes on a T4.

In [7]:
# Adjust --batch-size if you get out-of-memory errors.
# 16 is a safe starting point for a T4; try 32–72 on an L4.
!time python main.py --batch-size 72 --surya-ocr {VOLUME}


══════════════════════════════════════════════════════════════
  Pipeline: --surya-ocr
  Targets:  1
══════════════════════════════════════════════════════════════

[1/1] tulsa_1922
  uuid: None  |  slug: tulsa_1922  |  type: dir
  csv:  output/tulsa_1922/tulsa_1922.csv
  imgs: output/tulsa_1922/

  ── pipeline/run_surya_ocr.py
    $ /usr/bin/python3 /content/drive/Othercomputers/My_Mac/directory-pipeline/pipeline/run_surya_ocr.py output/tulsa_1922 --batch-size 72
Loading Surya models…
2026-05-26 03:21:27.711204: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-26 03:21:27.781506: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following ins

In [8]:
# Verify all pages were processed
from pathlib import Path

vol_path = Path(PIPELINE_DIR) / VOLUME
images = sorted(list(vol_path.glob("*.jpg")) + list(vol_path.glob("*.png")))
surya_files = sorted(vol_path.glob("*_surya.json"))

print(f"Page images:  {len(images)}")
print(f"Surya JSON:   {len(surya_files)}")

missing = [
    img.name for img in images
    if not (vol_path / (img.stem + "_surya.json")).exists()
]
if missing:
    print(f"\n⚠ {len(missing)} pages still missing Surya output:")
    for m in missing[:10]:
        print(f"  {m}")
    if len(missing) > 10:
        print(f"  ... and {len(missing) - 10} more")
    print("Re-run the cell above to process them.")
else:
    print("\n✓ All pages have Surya output — ready for alignment")

Page images:  933
Surya JSON:   933

✓ All pages have Surya output — ready for alignment


---
## Step 2: Align OCR — match text to bounding boxes

We now have two things for each page:
- **Gemini's text** — accurate transcription, in reading order, but no position info
- **Surya's boxes** — precise positions on the page, but no text content

Alignment connects them using a sequence-alignment algorithm, with city and state headings as anchor points. A second pass then tries to recover any lines missed in the first.

The output is a `*_aligned.json` file for each page — every matched line has both text and a bounding box. Lines that couldn't be matched appear in an `unmatched_gemini` list and won't get coordinates in the final CSV.

**Confidence levels** in the output:
- `"word"` — high-confidence automatic match
- `"line"` — moderate-confidence automatic match
- `"manual"` — you confirmed or corrected this in the review UI (Step 3)

**Expected time:** A few minutes for most volumes — much faster than Surya, and no GPU needed.

In [11]:
!time python main.py --align-ocr --model {MODEL} {VOLUME}


══════════════════════════════════════════════════════════════
  Pipeline: --align-ocr
  Targets:  1
══════════════════════════════════════════════════════════════

[1/1] tulsa_1922
  uuid: None  |  slug: tulsa_1922  |  type: dir
  csv:  output/tulsa_1922/tulsa_1922.csv
  imgs: output/tulsa_1922/

  ── pipeline/align_ocr.py
    $ /usr/bin/python3 /content/drive/Othercomputers/My_Mac/directory-pipeline/pipeline/align_ocr.py output/tulsa_1922 --model gemini-3.1-flash-lite
Aligning 933 image(s) — model=gemini-3.1-flash-lite, 4 worker(s)…
[0001/933] Done:    0003_p15020coll12:2860_gemini-3.1-flash-lite_aligned.json
[0002/933] Done:    0004_p15020coll12:2861_gemini-3.1-flash-lite_aligned.json
[0003/933] Done:    0002_p15020coll12:2859_gemini-3.1-flash-lite_aligned.json
[0004/933] Done:    0001_p15020coll12:2858_gemini-3.1-flash-lite_aligned.json
[0005/933] Done:    0006_p15020coll12:2863_gemini-3.1-flash-lite_aligned.json
[0006/933] Done:    0005_p15020coll12:2862_gemini-3.1-flash-lite_ali

In [ ]:
# Summarize alignment quality across the volume
import json
from pathlib import Path

vol_path = Path(PIPELINE_DIR) / VOLUME
aligned_files = sorted(vol_path.glob("*_aligned.json"))
print(f"Aligned JSON files: {len(aligned_files)}")

if aligned_files:
    total_matched = total_unmatched = flagged = 0
    problem_pages = []

    for af in aligned_files:
        data = json.loads(af.read_text())
        matched   = len(data.get("lines", []))
        unmatched = len(data.get("unmatched_gemini", []))
        flag      = data.get("possible_column_merge", False)
        total_matched   += matched
        total_unmatched += unmatched
        if flag:
            flagged += 1
        if matched > 5 and unmatched > matched * 0.5:
            problem_pages.append((af.name, matched, unmatched))

    total = total_matched + total_unmatched
    pct = total_matched / total * 100 if total else 0
    print(f"\nOverall match rate:             {total_matched}/{total} lines ({pct:.0f}%)")
    print(f"Pages flagged for column merge: {flagged}")

    if problem_pages:
        print(f"\nPages with >50% unmatched lines ({len(problem_pages)} total):")
        for name, m, u in sorted(problem_pages, key=lambda x: -x[2])[:10]:
            print(f"  {name}  matched={m}  unmatched={u}")
        print("\nThese are good candidates for manual review in Step 3.")
    else:
        print("\n✓ No pages with severe alignment problems detected")

---
## Step 3: Review Alignment — fix problem pages

Alignment works automatically on most pages, but some layouts trip it up — two-column text, unusual formatting, or pages where Surya missed an entire column. The review UI lets you spot-check flagged pages and manually correct bad matches.

**How the UI works:**
1. Pages are sorted by problem severity — flagged and high-unmatched pages come first
2. Each page shows the scan on the left and matched text on the right
3. Click to link a text line to a different bounding box, or confirm an existing match
4. Click **Done** to save your corrections and move to the next page

The review tool is a Flask server running inside your Colab session. Colab's built-in proxy exposes it to your browser — no account or extra setup needed.

> 💡 **You don't need to review every page.** Focus on pages the quality check above flagged. Good-enough alignment on most pages beats perfect alignment on a few.

In [ ]:
import subprocess
import time

# Stop any server already running on this port
subprocess.run(["fuser", "-k", f"{PORT}/tcp"], capture_output=True)
time.sleep(1)

# Start the review server in the background
subprocess.Popen(
    ["python", "-m", "pipeline.review_alignment",
     VOLUME,
     "--host", "0.0.0.0",
     "--port", str(PORT),
     "--model", MODEL],
    cwd=PIPELINE_DIR,
    stdout=open("/tmp/review.log", "w"),
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

print(f"Server starting on: {VOLUME}")
print("Surya models take ~30 seconds to load. Run the next cell to get the URL.")

In [ ]:
# Get the Colab proxy URL — no account or token needed
from google.colab.output import eval_js

url = eval_js(f"google.colab.kernel.proxyPort({PORT})")
print("✓ Review tool URL:")
print(f"  {url}")
print()
print("Open that link in a new tab.")
print("If it shows 'Bad Gateway', wait 10–15 seconds for the models to finish loading, then refresh.")

In [ ]:
# Optional: watch the server log to confirm it's up.
# Stop this cell (■) once you see "Models ready."
!tail -f /tmp/review.log

### Fallback: ngrok tunnel

If the Colab proxy URL shows a persistent error (not just a brief loading delay), ngrok is a reliable alternative.

1. Create a free account at [ngrok.com](https://ngrok.com) and copy your auth token
2. In the Colab left sidebar, open **Secrets** (🔑 icon)
3. Add a secret named `NGROK_TOKEN` with your token as the value
4. Run the cell below

In [ ]:
from pyngrok import ngrok
from google.colab import userdata

ngrok.set_auth_token(userdata.get("NGROK_TOKEN"))

# Disconnect any old tunnels first
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

public_url = ngrok.connect(PORT)
print("✓ Review tool URL (ngrok):")
print(f"  {public_url}")

---
## Step 4: Re-extract entries with bounding boxes

Once you've finished reviewing (or decided the automatic alignment is good enough), re-run entry extraction. This time the extractor reads the `*_aligned.json` files and attaches `#xywh=` coordinates to every entry it can match.

The resulting `canvas_fragment` URLs will point to the exact spot on the original page scan.

In [ ]:
# Re-run entry extraction to pick up the aligned bounding boxes.
# If you used a custom NER prompt from a different volume, add:
#   --ner-prompt output/<first-slug>/ner_prompt.md
!python main.py --extract-entries --model {MODEL} {VOLUME}

In [ ]:
# Check how many entries received bounding box coordinates
import csv
from pathlib import Path

vol_path = Path(PIPELINE_DIR) / VOLUME
csvs = sorted(vol_path.glob(f"entries_{MODEL}*.csv"))

if not csvs:
    print("No entries CSV found — check that extraction completed successfully")
else:
    csv_path = csvs[-1]
    with open(csv_path) as f:
        rows = list(csv.DictReader(f))

    with_bbox    = sum(1 for r in rows if "#xywh=" in r.get("canvas_fragment", ""))
    without_bbox = len(rows) - with_bbox
    pct = with_bbox / len(rows) * 100 if rows else 0

    print(f"CSV: {csv_path.name}")
    print(f"Total entries:        {len(rows)}")
    print(f"With bounding box:    {with_bbox}  ({pct:.0f}%)")
    print(f"Without bounding box: {without_bbox}")

    sample = next((r for r in rows if "#xywh=" in r.get("canvas_fragment", "")), rows[0] if rows else None)
    if sample:
        print("\nSample entry:")
        for k in ["name", "city", "state", "address", "canvas_fragment"]:
            val = sample.get(k, "")
            if val:
                print(f"  {k}: {val[:90]}{'…' if len(val) > 90 else ''}")

    if without_bbox > len(rows) * 0.2:
        print(f"\nNote: {without_bbox / len(rows):.0%} of entries have no bounding box.")
        print("Additional review in Step 3 can recover some of these.")

---
## Working with multiple volumes

The cells above process one volume at a time. If you have a large collection with many sub-volumes (e.g. `the_green_book_9ea5d5b0` has one UUID subdirectory per NYPL item), the widgets below let you step through them without editing the config cell each time.

**Surya across all sub-volumes** — runs Surya on each sub-volume sequentially.

**Review across sub-volumes** — lets you pick a sub-volume from a dropdown, start the review server, and get the URL — all without leaving the notebook.

### Run Surya across all sub-volumes

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path

# Point this at a collection directory that contains UUID sub-volumes.
# Example: output/the_green_book_9ea5d5b0
collection_input = widgets.Text(
    value="",
    placeholder="e.g. output/the_green_book_9ea5d5b0",
    description="Collection path:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="600px"),
)

batch_input = widgets.IntText(
    value=16,
    description="Batch size:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="200px"),
)

display(collection_input, batch_input)

In [ ]:
import subprocess
from pathlib import Path

COLLECTION_PATH = Path(PIPELINE_DIR) / collection_input.value.strip()
BATCH_SIZE = batch_input.value

sub_volumes = sorted(
    d for d in COLLECTION_PATH.iterdir()
    if d.is_dir() and not d.name.startswith("_")
)

print(f"Collection: {COLLECTION_PATH}")
print(f"Sub-volumes found: {len(sub_volumes)}")
print(f"Batch size: {BATCH_SIZE}")
print()

for i, vol in enumerate(sub_volumes, 1):
    images = list(vol.glob("*.jpg")) + list(vol.glob("*.png"))
    existing = list(vol.glob("*_surya.json"))
    remaining = len(images) - len(existing)
    print(f"[{i}/{len(sub_volumes)}] {vol.name}  ({remaining} pages remaining)")
    if remaining == 0:
        print("  ✓ Already complete, skipping")
        continue
    rel_path = vol.relative_to(Path(PIPELINE_DIR))
    result = subprocess.run(
        ["python", "main.py", "--batch-size", str(BATCH_SIZE), "--surya-ocr", str(rel_path)],
        cwd=PIPELINE_DIR,
    )
    if result.returncode != 0:
        print(f"  ⚠ Non-zero exit code — check output above")

print("\nDone.")

### Review alignment across sub-volumes

For large collections, loading the entire collection into the review tool at once is slow. Use this widget to step through one sub-volume at a time — pick a sub-volume, click **▶ Start Review**, then open the URL shown.

In [ ]:
import subprocess
import time
import re
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab.output import eval_js

_UUID_RE = re.compile(
    r'^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$', re.I
)

# ── Collection picker ─────────────────────────────────────────────────────────

OUTPUT_ROOT = Path(PIPELINE_DIR) / "output"

# Find all collection directories that contain UUID sub-volumes
collections = []
for top in sorted(OUTPUT_ROOT.iterdir()):
    if not top.is_dir() or top.name.startswith("_"):
        continue
    subs = [s for s in top.iterdir() if s.is_dir() and _UUID_RE.match(s.name)]
    if subs:
        collections.append(top)
    elif any((top / f).exists() for f in top.glob("*_aligned.json")):
        # Flat collection (no UUID subdirs) — treat as a single-item collection
        collections.append(top)

if not collections:
    print(f"No collections with aligned JSON found under {OUTPUT_ROOT}")
else:
    print(f"Collections found:")
    for c in collections:
        n = len(list(c.rglob("*_aligned.json")))
        print(f"  {c.name}  ({n} aligned JSON files)")

collection_dd = widgets.Dropdown(
    options=[(c.name, c) for c in collections],
    description="Collection:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="720px"),
)

subvol_dd = widgets.Dropdown(
    description="Sub-volume:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="720px"),
)

def _refresh_subvols(change=None):
    col = collection_dd.value
    if col is None:
        subvol_dd.options = []
        return
    subs = sorted(s for s in col.iterdir() if s.is_dir() and _UUID_RE.match(s.name))
    if subs:
        subvol_dd.options = [
            (f"{s.name}  ({len(list(s.rglob('*_aligned.json')))} aligned)", s)
            for s in subs
        ]
    else:
        # Flat collection — the collection itself is the target
        n = len(list(col.rglob("*_aligned.json")))
        subvol_dd.options = [(f"{col.name}  ({n} aligned)", col)]

collection_dd.observe(_refresh_subvols, names="value")
_refresh_subvols()

start_btn = widgets.Button(
    description="▶ Start Review",
    button_style="primary",
    layout=widgets.Layout(width="160px"),
)

_reviewed = set()
progress_label = widgets.Label()

def _update_progress():
    total = len(subvol_dd.options)
    progress_label.value = f"Reviewed this session: {len(_reviewed)}/{total}"

_update_progress()
status_out = widgets.Output()

def _on_start(_b):
    sub = subvol_dd.value
    if sub is None:
        with status_out:
            print("No sub-volume selected.")
        return
    with status_out:
        clear_output()
        print(f"Stopping any existing server on port {PORT}…")
        subprocess.run(["fuser", "-k", f"{PORT}/tcp"], capture_output=True)
        time.sleep(1)

        rel = sub.relative_to(Path(PIPELINE_DIR))
        n_aligned = len(list(sub.rglob("*_aligned.json")))
        print(f"Starting: {sub.name}")
        print(f"  {n_aligned} aligned JSON file(s)")

        subprocess.Popen(
            ["python", "-m", "pipeline.review_alignment",
             str(rel),
             "--host", "0.0.0.0",
             "--port", str(PORT),
             "--model", MODEL],
            cwd=PIPELINE_DIR,
            stdout=open("/tmp/review.log", "w"),
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )

        time.sleep(3)

        try:
            proxy_url = eval_js(f"google.colab.kernel.proxyPort({PORT})")
            print(f"\n✓ Review URL:\n  {proxy_url}\n")
        except Exception:
            print("(Colab proxy URL unavailable — run the ngrok fallback cell)")

        print("Check server log:  !cat /tmp/review.log | tail -20")

        _reviewed.add(sub.name)
        _update_progress()

start_btn.on_click(_on_start)

display(widgets.VBox([
    collection_dd,
    subvol_dd,
    widgets.HBox([start_btn, widgets.Label("   "), progress_label]),
    status_out,
]))

In [ ]:
# Ngrok fallback for the sub-volume review (use if Colab proxy doesn't work)
from pyngrok import ngrok
from google.colab import userdata

ngrok.set_auth_token(userdata.get("NGROK_TOKEN"))

for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

public_url = ngrok.connect(PORT)
print("✓ Review tool URL (ngrok):")
print(f"  {public_url}")

---
## Troubleshooting

**Surya runs out of memory (CUDA OOM)**  
Lower `--batch-size` (try `4` or `8`) and re-run. Surya skips pages that already have a `_surya.json`, so no work is repeated.

**Review server shows 'Bad Gateway'**  
The Surya models take ~30 seconds to load. Wait, then refresh. Check the log:
```python
!cat /tmp/review.log | tail -20
```

**Clicking Done in the review UI shuts down the server**  
This is intentional — Done saves your work. To continue reviewing more pages or a different sub-volume, re-run the Start server cell (or click ▶ Start Review again in the widget).

**`ERR_NGROK_324` — too many tunnels**  
Disconnect old ones then reconnect:
```python
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)
```

**Port already in use**  
```python
!fuser -k 5000/tcp
```
Then re-run the Start server cell.

**Colab session disconnects mid-Surya**  
Surya saves each page's output as it goes. Reconnect, re-run the Setup cells, and re-run the Surya cell — it will skip already-processed pages and pick up where it left off.